In [ ]:
import cv2
import os
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
recognizer = cv2.face.LBPHFaceRecognizer_create(
    radius=1, 
    neighbors=8, 
    grid_x=8, 
    grid_y=8
)

def get_data(path):
    image_paths = [os.path.join(path, f) for f in os.listdir(path)]
    faces = []
    ids = []
    
    # CLAHE helps to improve contrast locally (better than global equalization)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))

    for img_path in image_paths:
        try:
            # 1. Load image and convert to grayscale
            img = Image.open(img_path).convert('L') 
            img_np = np.array(img, 'uint8')
            
            # 2. Extract ID
            user_id = int(os.path.split(img_path)[-1].split(".")[1])
            
            # 3. Pre-processing: Smoothing + Contrast Enhancement
            # Bilateral filter removes noise but keeps edges (eyes/nose/mouth) sharp
            smoothed = cv2.bilateralFilter(img_np, 5, 75, 75)
            enhanced = clahe.apply(smoothed)
            
            # 4. Augmentation: Add the original processed image
            faces.append(enhanced)
            ids.append(user_id)
            
            # 5. Augmentation: Add a horizontally flipped version
            # This helps if the person is facing a slightly different way
            flipped_img = cv2.flip(enhanced, 1)
            faces.append(flipped_img)
            ids.append(user_id)
            
        except Exception as e:
            print(f"Skipping {img_path}: {e}")
            
    return faces, ids

# Load and Split
data_path = r'..\Dataset\training\Cleaned_Training'
all_faces, all_ids = get_data(data_path)

X_train, X_test, y_train, y_test = train_test_split(
    all_faces, all_ids, test_size=0.3, random_state=40
)

# Train
recognizer.train(X_train, np.array(y_train))
print(f"Model trained on {len(X_train)} images.")

# Save the model
model_path = r'..\trainer.yml'
recognizer.save(model_path)
print(f"Model saved to {model_path}")


# Map IDs to Names
id_to_name = {
    1: "Besheer\t",
    2: "Ashraf\t",
    3: "Seif\t",
    4: "Sallam\t",
    5: "Roger\t",
    6: "Omar\t"
}

# Evaluate
correct = 0
total = len(X_test)

for i in range(total):
    predicted_id, confidence = recognizer.predict(X_test[i])
    
    # IMPROVEMENT: Adding a confidence threshold
    # If confidence > 150, it's likely a bad match regardless of the ID
    if predicted_id == y_test[i] :
        correct += 1
        result = "MATCH"
    else:
        result = "WRONG"
        
    actual_name = id_to_name.get(y_test[i], "Unknown")
    predicted_name = id_to_name.get(predicted_id, "Unknown")
    print(f"Actual: {actual_name} | Predicted: {predicted_name} | Conf: {confidence:.2f} | {result}")

accuracy = (correct / total) * 100
print(f"\nFinal Accuracy: {accuracy:.2f}%")

Model trained on 189 images.
Model saved to ..\trainer.yml
Actual: Sallam	 | Predicted: Sallam	 | Conf: 122.33 | MATCH
Actual: Besheer	 | Predicted: Omar	 | Conf: 114.53 | WRONG
Actual: Seif	 | Predicted: Roger	 | Conf: 117.62 | WRONG
Actual: Roger	 | Predicted: Seif	 | Conf: 115.16 | WRONG
Actual: Ashraf	 | Predicted: Ashraf	 | Conf: 70.29 | MATCH
Actual: Omar	 | Predicted: Omar	 | Conf: 110.27 | MATCH
Actual: Sallam	 | Predicted: Ashraf	 | Conf: 125.96 | WRONG
Actual: Sallam	 | Predicted: Roger	 | Conf: 119.36 | WRONG
Actual: Seif	 | Predicted: Seif	 | Conf: 105.52 | MATCH
Actual: Sallam	 | Predicted: Besheer	 | Conf: 117.39 | WRONG
Actual: Sallam	 | Predicted: Sallam	 | Conf: 118.05 | MATCH
Actual: Seif	 | Predicted: Omar	 | Conf: 119.71 | WRONG
Actual: Besheer	 | Predicted: Besheer	 | Conf: 114.24 | MATCH
Actual: Omar	 | Predicted: Omar	 | Conf: 109.00 | MATCH
Actual: Sallam	 | Predicted: Sallam	 | Conf: 105.35 | MATCH
Actual: Besheer	 | Predicted: Besheer	 | Conf: 108.87 | MATCH
A